In [0]:
dbutils.widgets.text("catalog", "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_gold", "gold")

catalog = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

silver_customers_table = f"{catalog}.{schema_silver}.silver_customers"
silver_transactions_table = f"{catalog}.{schema_silver}.silver_transactions"
silver_accounts_table = f"{catalog}.{schema_silver}.silver_accounts"
silver_credit_table = f"{catalog}.{schema_silver}.silver_credit"
silver_branches_table = f"{catalog}.{schema_silver}.silver_branches"
gold_table_full = f"{catalog}.{schema_gold}.gold_branch_performance"

In [0]:
from pyspark.sql import functions as F

# customer_branch
customer_branch = spark.table(silver_customers_table) \
    .select("customer_id", "branch_code")

# account_agg
account_agg = spark.table(silver_accounts_table) \
    .groupBy("customer_id") \
    .agg(
        F.count("account_id").alias("total_accounts"),
        F.sum("balance").alias("total_balance")
    )

# txn_agg
accounts = spark.table(silver_accounts_table)
transactions = spark.table(silver_transactions_table)
txn_agg = transactions.join(accounts, transactions.account_id == accounts.account_id, "inner") \
    .groupBy("customer_id") \
    .agg(
        F.count("txn_id").alias("total_transactions"),
        F.sum("amount").alias("total_transaction_amount")
    )

# branch_performance
branches = spark.table(silver_branches_table)

branch_performance = branches \
    .join(customer_branch, branches.branch_code == customer_branch.branch_code, "left") \
    .join(account_agg, customer_branch.customer_id == account_agg.customer_id, "left") \
    .join(txn_agg, customer_branch.customer_id == txn_agg.customer_id, "left") \
    .groupBy(branches.branch_code, branches.branch_name) \
    .agg(
        F.countDistinct(customer_branch.customer_id).alias("total_customers"),
        F.sum(account_agg.total_accounts).alias("total_accounts"),
        F.sum(account_agg.total_balance).alias("total_deposits"),
        F.sum(txn_agg.total_transactions).alias("total_transactions"),
        F.sum(txn_agg.total_transaction_amount).alias("total_transaction_amount")
    )

branch_performance.write.format("delta").mode("overwrite").saveAsTable(gold_table_full)

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.gold_branch_performance
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))

In [0]:
%sql
SELECT * FROM banking.gold.branch_performance